# Urdu Orpheus — UrduTTS on Kaggle
Enable **Internet** and an **NVIDIA GPU**. Add the **HF_TOKEN** Kaggle Secret with write access to a private model repository. No dataset upload, Drive mount, or Kaggle Input is needed.

Edit the settings below, push `orpheus-ft-urdutts/` to GitHub, and import this notebook. Run with `TEST=True` first (20 training + 10 validation rows). After TEST PASSED, set `TEST=False` for one full epoch from the original Orpheus pretrained base. Use **Save Version → Save & Run All**.

Keep the same code commit, run ID and config to resume interruptions. For another epoch after completion, choose a NEW run ID and set `INIT_FROM_RUN` to the completed run. This follows the ASLP approach: finished-epoch weights continue with a fresh optimizer/schedule; interrupted epochs restore the full optimizer/scheduler/RNG state.

The dataset release is pinned. Native 44.1 kHz audio is converted to 24 kHz mono PCM16 without trimming; `transcription` supplies the Urdu text. `eval` becomes validation. Test is not downloaded or used. Source conversion, exclusions and retained hours are recorded. The release contains ASR-generated transcripts and no speaker IDs; listen to the reference samples and check their text before committing GPU time.
If secret loading fails, enable the existing HF_TOKEN secret for this new notebook in Edit mode → Add-ons → Secrets, then save a new version. HF_SECRET_NAME must match its label exactly. The previous notebook’s access is separate. Existing HF_TOKEN environment credentials are also supported.


Cell 1 — clone a fixed version

In [ ]:
import os, sys, subprocess, re
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orhpeus-ft-urdutts.git"
REPO_REF = "main"  # First launch: your pushed branch. Then pin the printed commit for all resumes.
PROJECT_FOLDER = "."  # This standalone GitHub repository contains the scripts at its root.
HF_REPO_ID = "hassan-31x/orpheus-urdutts-checkpoints"  # Your private Hugging Face MODEL repository.
HF_SECRET_NAME = "HF_TOKEN"  # Exact Kaggle secret LABEL used in your previous notebook; change if different.
RUN_ID_OVERRIDE = "urdutts-r32-lr1e4-seed3407-epoch1"
INIT_FROM_RUN = None  # Initial epoch starts from the base. Epoch 2: set the completed epoch-1 run ID + a NEW run ID above.
LEARNING_RATE = None  # None keeps 1e-4, matching ASLP. Keep fixed within an interrupted run.
TEST = True
TEST_TRAIN_CLIPS = 20  # Use 10 or 20 for a quick end-to-end check.
TEST_VALIDATION_CLIPS = 10
AUDIT_ONLY = False  # True performs CPU data preparation/audit and reference playback, without training.
SESSION_HOURS = 10.5  # Choose below the runtime limit shown in YOUR Kaggle account.
CLONE = Path("/kaggle/working/orpheus-urdutts-code")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must be username/repository")
def load_hf_token(secret_name="HF_TOKEN", environ=None, secret_client_factory=None):
    # Keep a valid explicit environment token; do not overwrite it with a failed Kaggle lookup.
    environ = os.environ if environ is None else environ
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = environ.get(name, "").strip()
        if value:
            environ["HF_TOKEN"] = value
            return "environment:" + name
    if not isinstance(secret_name, str) or not secret_name.strip():
        raise RuntimeError("HF_SECRET_NAME must be the exact label in Kaggle Add-ons > Secrets")
    try:
        if secret_client_factory is None:
            from kaggle_secrets import UserSecretsClient
            secret_client_factory = UserSecretsClient
        value = secret_client_factory().get_secret(secret_name)
    except Exception as exc:
        # Never display exception text: upstream responses can contain credentials.
        error_type = type(exc).__name__
        raise RuntimeError(
            "Kaggle could not read the Hugging Face secret (" + error_type + "). "
            "Open this notebook in Edit mode > Add-ons > Secrets. Add/select the secret "
            "whose label exactly matches HF_SECRET_NAME and ENABLE its access for THIS notebook. "
            "Secrets enabled for the ASLP notebook must also be enabled for this new notebook. "
            "Use a Hugging Face token with write access to " + HF_REPO_ID + ". "
            "Then Save Version > Save & Run All again. Do not paste the token into notebook code."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError("The selected Kaggle secret is empty; update its value in Add-ons > Secrets")
    environ["HF_TOKEN"] = value.strip()
    return "Kaggle Secrets"

credential_source = load_hf_token(HF_SECRET_NAME)
print("Hugging Face credential loaded from", credential_source, "(value hidden)")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
if not AUDIT_ONLY:
    try:
        gpu_list = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        gpu_list = ""
    if "GPU" not in gpu_list:
        raise RuntimeError("Enable an NVIDIA GPU in Kaggle Settings; check remaining GPU quota")
    print(gpu_list.strip())
if not CLONE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=CLONE, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CLONE, check=True)
REPO = CLONE / PROJECT_FOLDER if (CLONE / PROJECT_FOLDER).is_dir() else CLONE
for name in ("setup_kaggle.py", "finetune_urdutts.py", "urdutts_data.py", "configs/urdutts.json"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Missing {name}: push the UrduTTS folder and check REPO_REF/PROJECT_FOLDER")
PROJECT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE, text=True).strip()
print("Project commit (pin REPO_REF to this for resumes):", PROJECT_COMMIT)
early_check = subprocess.run([sys.executable, str(REPO / "check_hub.py"), "--repo", HF_REPO_ID], cwd=REPO)
if early_check.returncode == 2:
    raise RuntimeError("Hugging Face storage check failed; see redacted diagnostics above")
if early_check.returncode:
    print("Early Hub check unavailable in host Python; it will run in the installed training venv")

Cell 2 — install and preflight

In [ ]:
# Dedicated venv inherits Kaggle CUDA wheels; installs project dependencies every fresh session.
ENV = Path("/kaggle/working/orpheus-urdutts-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
subprocess.run([TRAIN_PYTHON, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=REPO, check=True)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)

Cell 3 — private credentials and config

In [ ]:
import json
cfg = json.loads((REPO / "configs/urdutts.json").read_text())
cfg["run_id"] = RUN_ID_OVERRIDE or cfg["run_id"]
cfg["init_from_run"] = INIT_FROM_RUN
if TEST:
    if TEST_TRAIN_CLIPS < 1 or TEST_VALIDATION_CLIPS < 1:
        raise ValueError("Test clip counts must be positive")
    cfg["run_id"] += f"-smoke-{TEST_TRAIN_CLIPS}-{TEST_VALIDATION_CLIPS}-{PROJECT_COMMIT[:7]}"
    cfg.update(test_train_clips=TEST_TRAIN_CLIPS, test_validation_clips=TEST_VALIDATION_CLIPS,
               save_steps=2, eval_steps=2, logging_steps=1, sample_count=1)
if LEARNING_RATE is not None:
    cfg["learning_rate"] = LEARNING_RATE
cfg.update(checkpoint_backend="huggingface", hf_repo_id=HF_REPO_ID, session_hours=SESSION_HOURS)
CONFIG = Path("/kaggle/working/urdutts-run-config.json")
CONFIG.write_text(json.dumps(cfg, indent=2))
RUN = Path(cfg["work_dir"]) / "runs" / cfg["run_id"]
print("Run:", cfg["run_id"], "| TEST:", TEST, "| initialization:", INIT_FROM_RUN or cfg["model_id"])
print("Data:", cfg["dataset_id"], "revision:", cfg["dataset_revision"])
subprocess.run([TRAIN_PYTHON, str(REPO / "finetune_urdutts.py"),
                "--config", str(CONFIG), "--mode", "storage-check"], cwd=REPO, check=True)

Cell 4A — recommended unattended saved run

In [ ]:
# Blocks until the audit/training process completes. Failures print diagnostics.
command = [TRAIN_PYTHON, "-u", str(REPO / "finetune_urdutts.py"), "--config", str(CONFIG)]
if AUDIT_ONLY:
    command += ["--mode", "audit"]
import time

CONSOLE = Path("/kaggle/working/urdutts_train_console.log")

def show_tail(lines):
    try:
        with CONSOLE.open("rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 200_000))
            print("\n".join(f.read().decode("utf-8", "replace").splitlines()[-lines:]), flush=True)
    except OSError:
        pass

# Training output goes to a FILE, never the notebook stream: a stalled Kaggle log
# pipeline can block a process that writes to it. Progress tail printed every 5 min;
# full log: /kaggle/working/urdutts_train_console.log (and monitoring/ on Hugging Face).
print("Training output ->", CONSOLE, flush=True)
try:
    with CONSOLE.open("ab") as console:
        process = subprocess.Popen(command, cwd=REPO, stdout=console, stderr=subprocess.STDOUT,
                                   stdin=subprocess.DEVNULL)
        last_tail = time.time()
        while process.poll() is None:
            time.sleep(15)
            if time.time() - last_tail >= 300:
                print(f"--- {time.strftime('%H:%M:%S')} last lines ---", flush=True)
                show_tail(6)
                last_tail = time.time()
    show_tail(60)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)
except subprocess.CalledProcessError:
    import shutil
    print("Free disk GiB:", round(shutil.disk_usage(RUN).free / 2**30, 2))
    path = RUN / "disk_budget.json"
    if path.exists():
        print("Disk budget:", path.read_text())
    print("Training process exited. The traceback above is the failure; audit exclusions are separate.")
    if TEST:
        print("TEST FAILED. Keep TEST=True and fix the error above before a full run.")
    raise
if TEST and not AUDIT_ONLY:
    final_status = json.loads((RUN / "status.json").read_text()) if (RUN / "status.json").exists() else {}
    backup_info = json.loads((RUN / "backup_status.json").read_text()) if (RUN / "backup_status.json").exists() else {}
    wavs = sorted((RUN / "samples").glob("step-*/*.wav"))
    print("Status:", final_status.get("status"), "step:", final_status.get("step"))
    print("Validation:", final_status.get("validation"))
    print("Samples generated:", len(wavs), "| adapter_final exists:", (RUN / "adapter_final").is_dir())
    print("Remote backup:", backup_info)
    import math
    val_loss = final_status.get("validation", {}).get("full_validation_loss")
    valid_loss = isinstance(val_loss, (int, float)) and math.isfinite(val_loss)
    checkpoints = list(RUN.glob("checkpoint-*/COMPLETE.json"))
    if (final_status.get("status") == "smoke_complete" and (RUN / "adapter_final/adapter_model.safetensors").is_file()
            and checkpoints and valid_loss and backup_info.get("remote_snapshot_current")):
        print("TEST PASSED: training, sealed checkpoint, finite validation loss, Hub backup and final adapter worked.")
        print("Synthesis samples:", len(wavs), "(inspect audio; synthesis quality is not established by this test).")
        print("Set TEST=False and Save & Run All for the full epoch from the original base.")
    else:
        raise RuntimeError("TEST INCOMPLETE: check status, validation and remote backup above before a full run")

Cell 5 — status, converted reference audio and generated samples

Reference playback also works after an audit-only run. Compare the Urdu transcript against the audio; the training smoke test cannot certify transcript accuracy or speech quality.

In [ ]:
from IPython.display import Audio, display

def optional_json(path):
    try:
        return json.loads(path.read_text()) if path.exists() else {}
    except (OSError, ValueError) as exc:
        print("Optional report unavailable:", path.name, type(exc).__name__)
        return {}

status_value = optional_json(RUN / "status.json")
print(status_value or ("Dataset audit only" if AUDIT_ONLY else "No training status recorded"))
for path in sorted((RUN / "samples").glob("step-*/*.wav"))[-3:]:
    print(path)
    try:
        display(Audio(filename=str(path)))
    except (OSError, ValueError) as exc:
        print("Optional playback unavailable:", type(exc).__name__)
backup = optional_json(RUN / "backup_status.json")
if backup:
    print("Remote backup:", backup)

# Play the exact converted references used by training (no extra dataset download).
source_report = RUN / "dataset_source.json"
if source_report.exists():
    source = optional_json(source_report)
    from hashlib import sha256
    recipe_bytes = json.dumps(source["recipe"], ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    data_root = Path(cfg["work_dir"]) / "datasets" / sha256(recipe_bytes).hexdigest()
    import csv
    with (data_root / "validation.csv").open(encoding="utf-8") as handle:
        references = list(csv.DictReader(handle))[:3]
    for row in references:
        print("Reference:", row["text"])
        display(Audio(filename=str(data_root / row["audio"])))

Cell 6 — optional synthesis from the final adapter

Set `GENERATE_FINAL_SAMPLE=True` after completion. For another epoch, set TEST=False, INIT_FROM_RUN to the finished full run ID, and RUN_ID_OVERRIDE to a new epoch ID; rerun the notebook. Do not point INIT_FROM_RUN at an interrupted or smoke run.

In [ ]:
GENERATE_FINAL_SAMPLE = False
if GENERATE_FINAL_SAMPLE and not AUDIT_ONLY and (RUN / "adapter_final").is_dir():
    output = RUN / "manual_sample.wav"
    subprocess.run([TRAIN_PYTHON, str(REPO / "synthesize.py"), "--adapter", str(RUN / "adapter_final"),
                    "--text", "آج موسم بہت خوشگوار ہے۔ ہم اردو بولنا سیکھ رہے ہیں۔", "--output", str(output)],
                   cwd=REPO, check=True)
    display(Audio(filename=str(output)))